# Header cleaning
In this notebook, we'll clean the fasta headers for our proteomes before running Orthofinder, so that the downstream gene trees are interpretable.

In [1]:
import pandas as pd
from tqdm import tqdm
from os import listdir
from Bio import SeqIO

In [2]:
run_date = '21Nov2025'

## Creating mappings
We're going to strip everything after the first space and prepend the species name with an underscore, and then check tom ake sure that ouput looks sensible.

The species from phytozome have the species name as the second item, howeer, it's not a full species name, so we'll read in the species list to make that mapping. The transcriptome-derived proteomes need to have their SRA ID's converted. The `bob_set` proteomes don't follow a pattern, so we'll manually create a mapping here.

### Phytozome

In [13]:
species_info_df = pd.read_csv('/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/species_info/species_info_full.csv')
species_info_df.head()

,abbreviated_species,species_gbif,species,common_name,original_source,thermotolerance,reference,notes
0,A. americanus,Acorus americanus,Acorus americanus,NaN,phytozome,NaN,NaN,NaN
1,A. coerulea,Aquilegia caerulea,Aquilegia caerulea,Colorado columbine,phytozome,NaN,NaN,NaN
2,A. comosus,Ananas comosus,Ananas comosus,pineapple,phytozome,NaN,NaN,NaN
3,A. gerardiHAP1,Andropogon gerardi,Andropogon gerardi,NaN,phytozome,NaN,NaN,NaN
4,A. halleri,Arabidopsis halleri,Arabidopsis halleri,NaN,phytozome,NaN,NaN,NaN


In [14]:
species_info_df['abbreviated_species'] = species_info_df['abbreviated_species'].str.replace('. ', '').str.replace(' ', '')
species_info_df['species'] = species_info_df['species'].str.replace(' ', '_')
species_info_df.head()

,abbreviated_species,species_gbif,species,common_name,original_source,thermotolerance,reference,notes
0,Aamericanus,Acorus americanus,Acorus_americanus,NaN,phytozome,NaN,NaN,NaN
1,Acoerulea,Aquilegia caerulea,Aquilegia_caerulea,Colorado columbine,phytozome,NaN,NaN,NaN
2,Acomosus,Ananas comosus,Ananas_comosus,pineapple,phytozome,NaN,NaN,NaN
3,AgerardiHAP1,Andropogon gerardi,Andropogon_gerardi,NaN,phytozome,NaN,NaN,NaN
4,Ahalleri,Arabidopsis halleri,Arabidopsis_halleri,NaN,phytozome,NaN,NaN,NaN


In [22]:
phytozome_mapping = species_info_df[['abbreviated_species', 'species']].set_index('abbreviated_species').to_dict()['species']

### SRA

In [32]:
sra_mapping_df = pd.read_csv('/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/sra_study_info/filtered_single_rep_studies_18Sep2025_manual_filtering.csv')
sra_mapping_df.head()

,organism_name,run_accession,len_reads,tissue,tissue_type,study_accession,study_title,experiment_accession,experiment_title,experiment_desc,...,subsrc_note,authority,library name,sample type,geo_loc,plant_id,geographic location,time,source population,tag
0,Adenophora stricta,SRR31592529,150,leaf-stem,NaN,SRP549143,This BioProject includes raw sequencing data f...,SRX26957374,leaf+stem RNA of Adenophora stricta,leaf+stem RNA of Adenophora stricta,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Avicennia marina,SRR12145926,150,Leaf,NaN,SRP270003,RNASeq to identify salt responsive genes from ...,SRX8666650,RNASeq to identify salt responsive genes from ...,RNASeq to identify salt responsive genes from ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Avicennia marina,SRR12146026,150,Leaf,NaN,SRP270005,Whole Transcriptome sequencing and analaysis i...,SRX8666750,RNASeq from Avicennia leaf tissue,RNASeq from Avicennia leaf tissue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Avicennia marina subsp. australasica,SRR18449588,100,leaf,NaN,SRP365449,Mangrove Genome Project,SRX14582333,Amarinaau_leaf,Amarinaau_leaf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Avicennia marina subsp. eucalyptifolia,SRR18449587,100,leaf,NaN,SRP365449,Mangrove Genome Project,SRX14582334,Amarinaeu_leaf,Amarinaeu_leaf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
sra_mapping_df['organism_name'] = sra_mapping_df['organism_name'].str.replace('. ', '_').str.replace(' ', '_')

# Nabbed this nifty bit from https://stackoverflow.com/a/66914118
g = sra_mapping_df.groupby('organism_name')
sra_mapping_df['organism_name'] += g.cumcount().add(1).astype(str).radd('_').mask(g['organism_name'].transform('count')==1,'')
sra_mapping_df.head()

,organism_name,run_accession,len_reads,tissue,tissue_type,study_accession,study_title,experiment_accession,experiment_title,experiment_desc,...,subsrc_note,authority,library name,sample type,geo_loc,plant_id,geographic location,time,source population,tag
0,Adenophora_stricta,SRR31592529,150,leaf-stem,NaN,SRP549143,This BioProject includes raw sequencing data f...,SRX26957374,leaf+stem RNA of Adenophora stricta,leaf+stem RNA of Adenophora stricta,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Avicennia_marina_1,SRR12145926,150,Leaf,NaN,SRP270003,RNASeq to identify salt responsive genes from ...,SRX8666650,RNASeq to identify salt responsive genes from ...,RNASeq to identify salt responsive genes from ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Avicennia_marina_2,SRR12146026,150,Leaf,NaN,SRP270005,Whole Transcriptome sequencing and analaysis i...,SRX8666750,RNASeq from Avicennia leaf tissue,RNASeq from Avicennia leaf tissue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Avicennia_marina_subsp_australasica,SRR18449588,100,leaf,NaN,SRP365449,Mangrove Genome Project,SRX14582333,Amarinaau_leaf,Amarinaau_leaf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Avicennia_marina_subsp_eucalyptifolia,SRR18449587,100,leaf,NaN,SRP365449,Mangrove Genome Project,SRX14582334,Amarinaeu_leaf,Amarinaeu_leaf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
sra_mapping = sra_mapping_df[['run_accession', 'organism_name']].set_index('run_accession').to_dict()['organism_name']

### Bob's proteomes

In [4]:
bob_mapping = {
    'Auxeprot1_GeneCatalog_proteins_20170909.aa.fasta': 'Auxenochlorella_protothecoides',
    'E_curvula.fa': 'Eragrostis_curvula',
    'L_subracemosa.fa':  'Lindernia_subracemosa',
    'Selaginella_tamariscina.protein.fasta': 'Selaginella_tamariscina',
    'translated_GCA_002076135.1_ASM207613v1_genomic.fna': 'Xerophyta_schlecteri',
    'Boea_hygrometrica_gffread.translated_cds.fa': 'Boea_hygrometrica',
    'E_nindensis.fa': 'Eragrostis_nindensis',
    'M_caffra.fa': 'M_caffra',
    'S_lepidophylla.fa': 'Selaginella_lepidophylla',
    'T_spicatus.fa': 'Trachypogon_spicatus',
    'Chabra1_GeneCatalog_proteins_20200803.aa.fasta': 'Chara_braunii',
    'E_tef_V3.1.proteins.fasta': 'Eragrostis_tef',
    'M_inflexa_TR_CDS_v1.1.fasta': 'Marchantia_inflexa',
    'S_pyramidalis.fa': 'Sporobolus_pyramidalis',
    'X_viscosa.fa': 'Xerophyta_viscosa',
    'CratV4.1.12-2-20.proteins.fasta': 'Craterostigma_plantagineum',
    'Klenit1_GeneCatalog_proteins_20200807.aa.fasta': 'Klebsormidium_nitens',
    'O_capense.fa': 'Oropetium_capense',
    'S_stapfianus.fa': 'Sporobolus_stapfianus',
    'Z_japonica.fa': 'Z_japonica',
    'E_coracana.fa': 'Eleusine_coracana',
    'L_brevidens.fa': 'Lindernia_brevidens',
    'Pyezoensis_gffread.translated_cds.fa': 'Pyropia_yezoensis',
    'T_minimus.fa': 'T_minimus'
}

## Cleaning names
We're also going to replace the file names with the abbreviated names so that the column headers in the output file are also cleaner. I also am going to remove the description field and put the identifier in the ID column only, to avoid stuff getting junked up later.

In [59]:
base_set = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/base_set'
base_set_cleaned = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/base_set_cleaned'
print('\nRenaming base set...')
for filename in tqdm(listdir(base_set)):

    # Read in the original fasta
    fasta_sequences = SeqIO.parse(open(f'{base_set}/{filename}'),'fasta')

    # Get the short name for dictionary mapping
    species = filename.split('_')[1]
    if species == 'prepended':
        species = filename.split('_')[2]
    
    # Go through and replace description field
    renamed_seqs = []
    for i, seq in enumerate(fasta_sequences):
        seq.id = phytozome_mapping[species] + '__' + seq.description.split(' ')[0]
        seq.description = ''
        renamed_seqs.append(seq)
        # Uncomment below if you'd like to manually inspect the patterns for renaming
        # if i == 0:
        #     print(f'Updated ID for species {species}: {seq.id}')
        
    outfile = f'{base_set_cleaned}/{species}_{run_date}.fasta'
    with open(outfile, "w") as output_handle:
        SeqIO.write(renamed_seqs, output_handle, "fasta")


Renaming base set...


100%|██████████| 64/64 [00:38<00:00,  1.67it/s]


In [64]:
add_set = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/add_set'
add_set_cleaned = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/add_set_cleaned'
print('\nRenaming add set...')
for filename in tqdm(listdir(add_set)):

    # Read in the original fasta
    fasta_sequences = SeqIO.parse(open(f'{add_set}/{filename}'),'fasta')

    # Get the short name for dictionary mapping
    if filename.split('_')[1] in phytozome_mapping.keys():
        species = phytozome_mapping[filename.split('_')[1]]
        if species == 'prepended':
            species = phytozome_mapping[filename.split('_')[2]]
    elif filename.split('_')[0] in sra_mapping.keys():
        species = sra_mapping[filename.split('_')[0]] 
    
    # Go through and replace description field
    renamed_seqs = []
    for i, seq in enumerate(fasta_sequences):
        seq.id = species + '__' + seq.description.split(' ')[0]
        seq.description = ''
        renamed_seqs.append(seq)
        # Uncomment below if you'd like to manually inspect the patterns for renaming
        # if i == 0:
        #     print(f'Updated ID for species {species}: {seq.id}')
        
    outfile = f'{add_set_cleaned}/{species}_{run_date}.fasta'
    with open(outfile, "w") as output_handle:
        SeqIO.write(renamed_seqs, output_handle, "fasta")


Renaming add set...


100%|██████████| 147/147 [01:52<00:00,  1.30it/s]


In [9]:
bob_set = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/bob_set'
# Using add_set as the out path to combine them
add_set_cleaned = '/mnt/research/Walker_Lab_Research/Serena_project_data/selection-under-heat_data/orthofinder/add_set_cleaned'
print('\nRenaming base set...')
for filename in tqdm(listdir(bob_set)):

    # Read in the original fasta
    fasta_sequences = SeqIO.parse(open(f'{bob_set}/{filename}'),'fasta')

    # Get species short name
    species = bob_mapping[filename]
    species_short = species.split('_')[0][0] + '_' + species.split('_')[1]

    # Go through and replace description field
    renamed_seqs = []
    for i, seq in enumerate(fasta_sequences):
        seq.id = species + '__' + seq.description.split(' ')[0]
        seq.description = ''
        renamed_seqs.append(seq)
        # Uncomment below if you'd like to manually inspect the patterns for renaming
        # if i == 0:
        #     print(f'Updated ID for species {species}: {seq.id}')
        
    outfile = f'{add_set_cleaned}/{species}_{run_date}.fasta'
    with open(outfile, "w") as output_handle:
        SeqIO.write(renamed_seqs, output_handle, "fasta")


Renaming base set...


100%|██████████| 24/24 [00:13<00:00,  1.75it/s]
